# **Multi-Asset Portfolio Performance, Risk & Attribution**

**FINM3422 — Assessment 2 - Group 8**

*Analysis Period: January 2016 – December 2025*

# **1.0 Introduction**

This report presents a performance, risk, and attribution analysis of a multi-asset fund investing across five asset-class sleevs:

- Australian Equities (AUS EQ)
- International Equities (INTL EQ)
- Bonds/ Fixed Income (BONDS)
- Real Estate (RE)
- Private Equity/ Venture Capital (PE/VC)

The analysis covers the 10-year performance from January 2016 to December 2025, using monthly return data for each sleeve's manager and corresponding benchmark index. 

# **2.0 Data Overview**
### Environment Info, Imports and Config

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../src")
from data_loader import load_data, validate_data
from performance import all_sleeves_summary, annualised_return, annualised_volatility, max_drawdown
from attribution import brinson_attribution

print(sys.version)
print("pandas:", pd.__version__, "numpy:", np.__version__)
pd.set_option("display.float_format", "{:.4f}".format)

### Parameters and Paths

In [ ]:
# Base paths
BASE_path = "/Users/ameliaperry/Desktop/BAFE/FINM3422/python case/FINM3422-Group-8-Task-2-1/data/"

# TAA weights
taa_weights = pd.Series({
    "aus_eq": 0.35, 
    "intl_eq": 0.35, 
    "bonds": 0.15, 
    "re": 0.05, 
    "pevc": 0.10})

### Data Load and Sanity Check

In [ ]:
# Load and Validate Data
df_managers, df_benchmarks, df_rf, df_saa = load_data(BASE_path)
validate_data(df_managers, df_benchmarks, df_rf)
saa_weights = df_saa["Weight"]

# Sanity Checks
print("Any nulls in benchmarks?", df_benchmarks.isna().sum().sum())
print("Any nulls in managers?", df_managers.isna().sum().sum())
print("Dates aligned?", df_benchmarks.index.equals(df_managers.index))
print("Monotonic dates?", df_benchmarks.index.is_monotonic_increasing)
display(df_managers.head())
display(df_benchmarks.head())
display(df_rf.head()) 
display(df_saa.head())

data loaded successfully across all five sleeves for both managers and benchmarks, covering 120 monthly observations from January 2016 to December 2025. All series are fully aligned with no missing values, assuring the data is ready to be analysed. 

# **3.0 Performance and Risk Analysis**

### Methodology

**Annualised Return** — geometric growth rate of monthly returns:
$$R_{\text{ann}} = \left(\prod_{t=1}^{n}(1 + r_t)\right)^{12/n} - 1$$
- $r_t$: monthly return in month $t$
- $n$: number of monthly observations

**Annualised Volatility** — monthly standard deviation scaled annually:
$$\sigma_{\text{ann}} = \sigma_{\text{monthly}} \times \sqrt{12}$$

**Sharpe Ratio** — return per unit of total risk:
$$\text{Sharpe} = \frac{R_{\text{ann}} - R_{f,\text{ann}}}{\sigma_{\text{ann}}}$$
- $R_{f,\text{ann}}$: annualised risk-free rate
- $\sigma_{\text{ann}}$: annualised volatility

**Tracking Error** — volatility of active returns vs benchmark:
$$TE = \sigma(r_P - r_B) \times \sqrt{12}$$
- $r_P - r_B$: monthly active return

**Information Ratio** — active return per unit of active risk:
$$IR = \frac{R_P - R_B}{TE}$$

**Maximum Drawdown** — largest peak-to-trough decline:
$$MDD = \min_t \left(\frac{W_t - \max(W)}{\max(W)}\right)$$
- $W_t$: cumulative wealth at time $t$


### Results Summary & Display

In [ ]:
summary = all_sleeves_summary(df_managers, df_benchmarks, df_rf)

pct_rows = [
    "Annualised Return (Manager)",
    "Annualised Return (Benchmark)",
    "Annualised Volatility",
    "Tracking Error",
    "Maximum Drawdown (Manager)",
    "Maximum Drawdown (Benchmark)",
]
ratio_rows = ["Sharpe Ratio", "Information Ratio"]

print("Performance & Risk Summary (%)")
display(summary.loc[pct_rows].T.map(lambda x: f"{x:.2%}"))

print("Ratios")
display(summary.loc[ratio_rows].T.map(lambda x: f"{x:.2f}"))

Accross the five sleeves, manager returns were consistently above the relative benchmarks, suggesting overall positive active management over the full 10-year sample period. However, the level and consistency of outperformance varied significantly between sleeves. 

**Australian Equities** showed modest outperformance (6.91% vs 5.88%), with a low Sharpe index of 0.27, reflective of high volatility compared to return. Maximum drawdown of -24.63% is consistent with the significant Australian equity market fluctuations over this time. 

**International Equities** performed the strongest on both an absolute and risk-adjusted returns basis. The manager delivered 15.57% annualsied against a 13.94% benchmark while maintaining the lowest volatility of the equity sleeves at 11.96%. The Sharpe ratio of 1.05 is the only sleeve to achieve above 1.0, indicating genuine risk-adjusted perfromance. The low tracking error of 2.57% alongside an IR of 0.64 suggests the manager was disciplined and actve by consistently added value without taking large bets.

**Bonds** produced near-zero risk-adjusted returns, with the Sharpe Ratio of -0.00 reflecting that the return of 3.02% was only minorly above the risk-free rate over this period. This was to be expected, as the firs half of the sample period's low-yield environment compressed bond yields. The 2022 rate hiking cycle generated significant mark-to-market losses, reflectd in the -11.35 maximum drawdown. Despite this, the bonds manager produced the highest IR of 1.21 across all sleeves, outfperforming the benchmark by 1.13% with a tracking error of only 0.94%. Even though bonds achieved the lowest return, the manager for this sleeve was the most effective by adding substantial value relative to the benchmark despite the challenging rate environment.

**Real Estate** produced the second highest manager return at 10.84% value versus the benchmark of 8.03%, an outperformance of 2.81%. However, this also had the highest volatility of 18.72% and largest manager drawdown of -28.69% across akk sleeves. The tracking error almost three times larger than international equities of 7.10% indicates the manager took significant active risk to generate this return, also reflected in the relatively low IR of 0.39. The benchmarks lower drawdown of -37.41% suggests the manager navigated downside risk during periods of market stress, which is the primary source of outperformance. 

**PE/VC** delivered 10.24% annualised return with the lowest volatility of the sleeves at 10.86% and the second highest Sharpe ratio of 0.66. This smooth return profile is typical of private market assets, where valuations are reported infrequently and do not always accurately capture short-term market movements. The low tracking error of 2.08% and IR of 0.41 suggest steady, consistent outperformance rather than large active tilts.


### Visualisations

In [ ]:
# Plots 1 to 5 - Cumulative Returns per sleeve
# manager vs benchmark
for sleeve in df_managers.columns:
    wealth_mgr = (1 + df_managers[sleeve]).cumprod()
    wealth_bm  = (1 + df_benchmarks[sleeve]).cumprod()
    plt.figure(figsize=(10, 4))
    plt.plot(wealth_mgr, label="Manager")
    plt.plot(wealth_bm, label="Benchmark")
    plt.title(sleeve.upper() + " Cumulative Returns")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.legend()
    plt.grid(True)
    plt.show()

# Plot 6 - Sharpe Ratios across all sleeves
sleeves = list(df_managers.columns)
sharpe_ratios = [summary.loc["Sharpe Ratio", sleeve] for sleeve in sleeves]

plt.figure(figsize=(8, 4))
plt.bar(sleeves, sharpe_ratios)
plt.title("Sharpe Ratios by Sleeve")
plt.ylabel("Sharpe Ratio")
plt.show()

#Plot 7 - Information Ratios across all sleeves
sleeves = list(df_managers.columns)
ir_ratios = [summary.loc["Information Ratio", sleeve] for sleeve in sleeves]

plt.figure(figsize=(8, 4))
plt.bar(sleeves, ir_ratios)
plt.title("Information Ratios by Sleeve")
plt.ylabel("Information Ratio")
plt.show()

The cumulative return charts reinforce the summary table findings and provide further context on the timing and consistency of outperformance.

**Equity sleeves (AUS EQ & INTL EQ)** both show a sharp drawdown in early 2020 consistent with the COVID-19 shock, followed by a quicker recovery by the manager than the benchmark. INTL EQ shows the largest divergence, with the manager accelerating well above the benchmark from 2023 onwards to reach $4.20 versus $3.70 by 2026, the widest  terminal gap of all five sleeves and the primary driver of its 15.57% annualised return.

**Real Estate** experienced the most severe benchmark drawdown in 2020, falling back to approximately $1.00 while the manager held above $1.50. This justifies the large difference in maximum drawdown (-28.69% vs -37.41%). Both recovered strongly after 2021, with the manager sustaining a wide and growing gap through to 2026.

**Bonds and PE/VC** both show smooth, steady growth with the manager's returns consistently above the benchmark throughout the entire period and no visible severe drawdowns. The Bonds gap widens significantly after 2022, consistent with the manager navigating the rate hiking cycle more effectively than the benchmark.

**Sharpe and IR charts** confirm that International Equities and PE/VC are the strongest risk-adjusted performers on both measures, while AUS EQ is low on both. The Bonds IR standing well above all other sleeves despite its near-zero Sharpe highlights that absolute and relative performance metrics are highly relevant on the manager's skill.

# **4.0 APRA-Inspired Performance and Risk Checks**

### TAA Weights and Total Fund Return

In [ ]:
#Total Fund Monthly Returns
# weighted average of manager returns using TAA weights
df_fund_monthly = df_managers.mul(taa_weights).sum(axis=1)

# Total Benchmark Monthly Returns
# weighted average of benchmark returns using SAA weights
saa_weights = df_saa["Weight"]
df_benchmark_monthly = df_benchmarks.mul(saa_weights, axis=1).sum(axis=1)

print("Fund monthly returns (first 5):")
display(df_fund_monthly.head())

### APRA Checks

In [ ]:
# Inputs
CPI_TARGET = 0.025 # assumed CPI of 2.5%
RETURN_PREMIUM = 0.035 # difference between CPI and return target
RETURN_TARGET = CPI_TARGET + RETURN_PREMIUM # 6% return target
VOL_LIMIT = 0.10 # 10% annualised volatility limit
DRAWDOWN_LIMIT = -0.25 # -25% maximum drawdown limit

# Fund Level Metrics
fund_ann_return = annualised_return(df_fund_monthly)
fund_ann_vol = annualised_volatility(df_fund_monthly)
fund_mdd = max_drawdown(df_fund_monthly)

# Check 1: Long-term return vs target
check1_pass = fund_ann_return >= RETURN_TARGET

# Check 2: Max Drawdown vs -25% Limit
check2_pass = fund_mdd >= DRAWDOWN_LIMIT

# Check 3: Volatility vs 10% Limit
check3_pass = fund_ann_vol <= VOL_LIMIT

# Results
APRA_results = pd.DataFrame({
    "Check": ["Long-run return >= CPI + 3.5% (6.0%)", "Max Drawdown >= -25%", "Annualised Volatility <= 10%"],
    "Fund Value": [f"{fund_ann_return:.2%}", f"{fund_mdd:.2%}", f"{fund_ann_vol:.2%}"],
    "Threshold": [f"{RETURN_TARGET:.2%}",f"{DRAWDOWN_LIMIT:.2%}", f"{VOL_LIMIT:.2%}"],
    "Pass / Fail": ["Pass" if check1_pass else "Fail", "Pass" if check2_pass else "Fail", "Pass" if check3_pass else "Fail"]})

print("APRA Performance & Risk Checks:")
display(APRA_results)

The fund passed all three APRA-inspired checks over the full sample period. The long-run return of 10.38% greatly exceed the CPI + 3.5% target of 6.00%, suggesting the fund delivered strong real returns well above what would be needed to support the business's retirement payment obligations. This margin of 4.38% above the target means the fund is not just relying on favourable short-term conditions. 

The maximum drawdown of -9.22% passed the -25% threshold easily. This is a relatively shallow drawdown for a growth-oriented multi-asset fund that included the COVID-19 market disruption and 2022 rate-hiking cycle. The diversifications across five asset classes contributed to containing the total fund's decline in these periods, particularly the inclusion of Bonds and PE/VC which experienced the smallest drawdowns.

Annualised volatility of 6.85% sits within the 8-12% band typical of a balanced superannuation fund, and below the 10% limit. This is lower than to be expected given the 70% combined equity weight, and reflects the cushioning effect of the Bonds and PE/VC sleeves on the overall portfolio risk through diversification. 

### Shock Scenario A

20% Equity Crash in one month

In [ ]:
# Apply shock to returns in month recent months
shock_returns_a = df_managers.iloc[-1].copy() # use last month returns as base
shock_returns_a["aus_eq"] = -0.20 # apply -20% shock to Australian equity
shock_returns_a["intl_eq"] = -0.20 # apply -20% shock to International equity
# Bonds, RE, PEVC assumed unaffected in shock scenario (equities only)

# Calculate shocked fund return for that month
return_shock_a = (taa_weights * shock_returns_a).sum()

# Comparison to normal return
return_normal_a = (taa_weights * df_managers.iloc[-1]).sum()

print("Shock Scenario: Equity Crash (-20% AUS EQ & INTL EQ in one month)")
shock_scenario_a_results = pd.DataFrame({
    "Scenario": ["Normal", "Equity Crash Shock"],
    "Portfolio Return": [f"{return_normal_a:.2%}", f"{return_shock_a:.2%}"],
    "AUS EQ Return": [f"{df_managers.iloc[-1]['aus_eq']:.2%}", "-20.00%"],
    "INTL EQ Return": [f"{df_managers.iloc[-1]['intl_eq']:.2%}", "-20.00%"]})
display(shock_scenario_a_results)
print(f"\nShock Impact on Portfolio Returns: {return_shock_a - return_normal_a:.2%}")

A simultaneous -20% monthly shock to both Australian and International Equities produces a portfolio return of -13.73% for that month, compared to a normal return of 1.27%, a 15.01% reduction. This is a direct consequence of the fund's 70% combined Equities TAA weight, that means equity market shocks have a proportionally large impact on the total fund. The result 
demonstrates that while the fund's diversification across bonds, real estate, and PE/VC provides some buffer, being predominantly equity-driven means the fund would experience significant short-term losses in a severe equity market disruption.

### Shock Scenario B - Bond Yield Spike

bond yields increase 150bps

In [ ]:
# bonds: -(duration x 1.5%) = -5%
# REITS: -5%
# Equities: -2%
# PE/VC: -2%

shock_returns_b = df_managers.iloc[-1].copy()
shock_returns_b["aus_eq"] = -0.02
shock_returns_b["intl_eq"] = -0.02
shock_returns_b["bonds"] = -0.05
shock_returns_b["re"] = -0.05
shock_returns_b["pevc"] = -0.02

return_shock_b = (taa_weights * shock_returns_b).sum()
return_normal_b = (taa_weights * df_managers.iloc[-1]).sum()

print("Shock Scenario B: Bond Yield Spike")
shock_scenario_b_results = pd.DataFrame({
    "Scenario": ["Normal (last month)", "Bond Yield Spike"],
    "AUS EQ": [f"{df_managers.iloc[-1]['aus_eq']:.2%}", "-2.00%"],
    "INTL EQ": [f"{df_managers.iloc[-1]['intl_eq']:.2%}", "-2.00%"],
    "Bonds": [f"{df_managers.iloc[-1]['bonds']:.2%}", "-5.00%"],
    "RE": [f"{df_managers.iloc[-1]['re']:.2%}", "-5.00%"],
    "PEVC": [f"{df_managers.iloc[-1]['pevc']:.2%}", "-2.00%"],
    "Portfolio Return": [f"{return_normal_b:.2%}", f"{return_shock_b:.2%}"]})
display(shock_scenario_b_results)
print(f"\nShock Impact on Portfolio Returns: {return_shock_b - return_normal_b:.2%}")

The bond yield spike scenario produces a portfolio return of -2.60% against a normal return of 1.27% , a 3.87% reduction. This is significantly less severe than the equity crash scenario, despite applying decreases in returns across all five asset classes. This occurs due to the TAA weighs in the rate-sensitive assets being relatively small (Bonds at 15% weight, and RE at 5%). Additionally, the equity shock in this scenario of -2% is much less sever than the -20% in Scenario A. The cross-asset nature fo the shock, where rising rates affect bonds, real estate, and equities simulataneously, illustrates the interconnected risk of a rate-driven selloff. However, the funds 20% fixed income allocation limits the total damage to reutnrs. This is consistent with the low overall volatility in the fund of 6.85%. 

### Total Fund Vs. Benchmark Comparison

In [ ]:
# Summary Comparison Table
fund_vol = annualised_volatility(df_fund_monthly)
bm_return = annualised_return(df_benchmark_monthly)
bm_vol = annualised_volatility(df_benchmark_monthly)
bm_mdd = max_drawdown(df_benchmark_monthly)

comparison = pd.DataFrame({
    "Metric": ["Annualised Return", "Annualised Volatility", "Maximum Drawdown"],
    "Total Fund (TAA)": [f"{fund_ann_return:.2%}", f"{fund_vol:.2%}", f"{fund_mdd:.2%}"],
    "Composite Benchmark (SAA)": [f"{bm_return:.2%}", f"{bm_vol:.2%}", f"{bm_mdd:.2%}"]
})
print("Total Fund vs Composite Benchmark Performance:")
display(comparison)

The total fund (TAA-weighted) outperformed the composite benchmark (SAA-weighted) across all three metric. The fund delivered an annualised return of 10.38% for the benchmark, while simultaneously achieving lower volatility (6.85% vs. 6.97%) and a less severe maximum drawdown (-9.22 vs -11.90)

The favourable outcome of generating higher returns with lower risk reflects the combined benefit of overweighting high-performance sleeves (INTL EQ and PE/VC) and the positive selection effects generated by managers across all five sleeves. These two effects are analysed in Section 5. 

# **5.0 Multi-Asset Performance Attribution**

### Formulas and Methodology

### Imports and Attribution

In [ ]:
# Check saa_weights index match df_managers columns
saa_weights = df_saa["Weight"]
saa_weights.index = saa_weights.index.astype(str).str.lower()

# Full Sample Attribution
df_attrib = brinson_attribution(df_managers, df_benchmarks, taa_weights, saa_weights)
print("Brinson Attribution (Full Sample):")
display(df_attrib.map(lambda x: f"{x:.2%}"))

The Brinson attribution reveals that both allocation and selection decisions contributed positively to the fund's 2.13% annualised active return, with much higher relative selection effects.

Selection effects totalled 12.15%, indicating that manager skill within each asset class was the primary driver of outperformance. Every sleeve generated a positive selection effect, supporting that all five external managers outperformed their respective benchmarks (after adjusting for benchmark weight). International Equities (4.42%) and Bonds (2.18%) had the higest selection contributions, reflecting consistent manager outperformance in these sleeves throughout the period.

Allocation effects totalled 7.39%, driven primarily by overweighting International Equities relative to SAA (35% TAA vs 30% SAA). This decision contributed 6.90% to the total allocation effect, as INTL EQ was the strongest performing benchmark over the period and overweighting this class added substantial value. PE/VC was the second-largest positive allocation contributor (4.82%), where again a sleeve with strong benchmark return was overweighted (5% to 10%)

The negative allocation effects in AUS EQ (-3.34%) and Bonds (-0.99%) reflect the cost of underweighting these sleeves relative to SAA. AUS EQ was underweighted by 5% (35% TAA vs 40% SAA) and still delivered positive benchmark returns, meaning the underweight decision reduced performance, whilst the manager still outperformed the benchmark through selection. This highlights how a manager can add value through skill (positive selection) while the allocation decision simultaneously detracts (negative allocation).

As the Real Estate weights were the same across both TAA and SAA, the allocation effect is 0.00%, and all of RE's 0.99% total contribution came from selection.

In [ ]:
# Bar Chart of Attribution
sleeves = [s for s in df_attrib.index if s != "total"]
alloc  = [df_attrib.loc[s, "Allocation Effect"] for s in sleeves]
select = [df_attrib.loc[s, "Selection Effect"]  for s in sleeves]

x = range(len(sleeves))
plt.figure(figsize=(10, 5))
plt.bar([i - 0.2 for i in x], alloc,  width=0.4, label="allocation effect", color="blue")
plt.bar([i + 0.2 for i in x], select, width=0.4, label="selection effect", color="orange")
plt.xticks(x, sleeves)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Brinson Attribution: Allocation vs Selection Effect by Sleeve")
plt.ylabel("Contribution to Active Return (full-sample sum)")
plt.legend()
plt.show()

The bar chart depicts the dominant selection effect across most sleeves, with International Equities and Bonds producing the largest selection bars. The allocation effect for International Equities is the highest bar effect across effects and sleeves, supporting it as the most impactful active manager decision in the portfolio. The negative allocation bars for AUS EQ and Bonds are more than offset the positiive allocation effect bars for both cases. PE/VC's allocation (4.82%) significantly exceeds selection (0.36%), suggesting the value added came primarily from the decision to overweight the asset class rather than from manager skill within it.

### Verify Active Return

In [ ]:
# total active return = fund return - composite benmark return
total_fund_ann = annualised_return(df_fund_monthly)
total_bench_ann = annualised_return(df_benchmark_monthly)
active_return = total_fund_ann - total_bench_ann

print(f"Total Fund Annualised Return: {total_fund_ann:.4%}")
print(f"Composite Benchmark Annualised Return: {total_bench_ann:.4%}")
print(f"Total Active Return: {active_return:.4%}")
print(f"Attribution Total (sum of allocation & selection effects): {df_attrib.loc['Total', 'Total Attribution']:.4%}")

The independently computed active return of 2.13% differs from the attribution 
total of 19.54% as the attribution effects are computed as cumulative monthly sums rather than annualised figures. Both confirm that the fund generated significant positive active return over the 10-year sample period through a combination of allocation and selection decisions.

# **6.0 Conclusion**

### Key Insights

Over the January 2016 to December 2025 sample period, the fund delivered 10.38% annualised against a composite benchmark of 8.24%, generating 2.14% active return while simultaneously achieving lower volatility (6.85% vs 6.97%) and a shallower maximum drawdown (-9.22% vs -11.90%). The fund passed all three APRA-inspired checks, with the return target exceeded by over 4%, drawdown within the -25% threshold, and volatility consistent with a balanced superannuation mandate.

Manager selection was the dominant driver of active return, contributing 12.15% versus 7.39% from allocation decisions. All five managers generated positive selection effects, with the Bonds manager producing the highest IR of 1.21 despite near-zero absolute returns.

International Equities was the single largest contributor to overall fund  performance, adding value through both an overweight allocation decision (TAA 35% vs SAA 30%) and strong manager outperformance, generating a total attribution of 11.32%. The shock scenario analysis identified the 
fund's vulnerability to equity crashes ficen the 70% combined equity weight producing a -13.73% monthly return, significantly larger than the -2.60% impact of a bond yield spike.

 ### Limitations

- **Constant weights:** Fixed TAA and SAA weights throughout the period do not reflect the continuous adjustments that occur in practice 

- **Attribution methodology:** The cumulative monthly attribution total of 19.54% is not directly comparable to the annualised active return of 2.13% due to differences in compounding. A rigorous attribution would apply geometric linking to produce annualised-consistent figures.

- **Simplified APRA checks:** The approximations used in this model do not reproduce the requirements of APRA Standard.

- **No costs or fees:** All returns are gross do not factor transaction costs and management fees, which would reduce reported outperformance.

